### 1 - Basic Edge Detection

Just a basic edge detection algorithm

In [13]:
import cv2

img = cv2.imread('test_img_ralsei1.jpeg')               # load image as a matrix of pixels (BGR order)
img = cv2.resize(img, (600, 800))                  # resize image
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)        # color space conversion

# convolution-based blur with 7x7 kernel size (a higher kernel size like 11x11 means a stronger blur)
# kernel size can only be odd, so the center pixel can still be created
blurred = cv2.GaussianBlur(gray, (7,7), 0)
edges = cv2.Canny(blurred, 50, 150)                 # edge detection (canny)

cv2.imshow('Image', edges)
cv2.waitKey(0)

-1

### 2 - Reading video

A simple script to read from a video file frame-by-frame

In [4]:
import cv2
capture = cv2.VideoCapture("test_vid_deltarune.mp4")

def rescaleFrame(frame, scale=0.25):
    # works for any media type (e.g. video, image, live video)
    width = int(frame.shape[1] * scale)
    height = int(frame.shape[0] * scale)

    newRes = (width, height)
    newImg = cv2.resize(frame, newRes, interpolation=cv2.INTER_AREA) # add interpolation to smooth out the edges between pixels
    return newImg

def changeRes(frame, width, height):
    # only works for live video (e.g. coming from a webcam)
    WIDTH_ID = 3
    HEIGHT_ID = 4
    frame.set(WIDTH_ID, width)
    frame.set(HEIGHT_ID, height)

while True:
    isTrue, frame = capture.read()
    frameResized = rescaleFrame(frame, 0.75)

    cv2.imshow("Video", frame)
    # cv2.imshow("Video", frameResized)

    if cv2.waitKey(20) & 0xFF == ord('d'): # press d to exit the simulation
        break

capture.release()
cv2.destroyAllWindows()


### 3 - Rescaling frames

Just a simple script to resize frames in an image

In [5]:
import cv2

img = cv2.imread("test_kris.webp")

def rescaleFrame(frame, scale=0.25):
    width = int(frame.shape[1] * scale)
    height = int(frame.shape[0] * scale)

    newRes = (width, height)
    newImg = cv2.resize(frame, newRes)
    return newImg

img = rescaleFrame(img, 0.5)

cv2.imshow("Kris Dreemurr", img)

cv2.waitKey(0)

-1

### 4 - Drawing shapes

In [33]:
import cv2
import numpy as np

def rescaleFrame(frame, scale=0.25):
    width = int(frame.shape[1] * scale)
    height = int(frame.shape[0] * scale)

    newRes = (width, height)
    newImg = cv2.resize(frame, newRes)
    return newImg


# (500, 500, 3) - create a 500x500 image with 3 colour channels (width, height, channels)
# create a matrix of zeros treated as a blank image
blankImg = np.zeros((500,500,3), dtype="uint8") 

# 1. paint a region of the image
blankImg[0:50, 100:300] = (255,255,0) # BGR (blue, green, red) channel. paint the image from y=0 to y=50 and x=100 to x=300

# 2. drawing a rectangle (pick two points)
cv2.rectangle(blankImg, (125,125), (375,375), (0,255,0), thickness=2)

# 3. draw a circle
cv2.circle(blankImg, (blankImg.shape[1] // 2, blankImg.shape[0] // 4), 80, (0,255,255), thickness=5)

# 4. draw a line
cv2.line(blankImg, (125,125), (400,400), (255, 255, 255), thickness=2)

# 5. write text on image
cv2.putText(blankImg, "hello!!!!!", (100,100), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (128,128,255), 2)

# img = cv2.imread("test_img_ralsei1.jpeg")
# img = rescaleFrame(img, 0.5)

cv2.imshow("Blank image", blankImg)
# cv2.imshow("Ralsei", img)

cv2.waitKey(0)

-1

### 5 - Essential functions

In [ ]:
import cv2

def rescaleFrame(frame, scale=0.5):
    width = int(frame.shape[1] * scale)
    height = int(frame.shape[0] * scale)

    newRes = (width, height)
    newImg = cv2.resize(frame, newRes) # resize an image with the new specified resolution
    return newImg

img = cv2.imread("test_img_city1.jpg")
img = rescaleFrame(img)

# greyscale image
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# blur - box blur under the hood - can have even sizes as it just 
# blur = cv2.blur(img, (20,20))
# gaussian blur - smoother blurs, less boxy
blur = cv2.GaussianBlur(img, (39,39), 0)

# edge cascade - find edges present in the image
# has 2 threshold values
# values above upper threshold = strong edges
# values below lower threshold = discarded (they're not edges)
# values in between = only kept if they connect to stronger edges
edges = cv2.Canny(img, 125, 250)

# dilate image - expand the edges in the image
dilated = cv2.dilate(edges, (3,3), iterations=3)

# erode image - thin out the edge pixels in the image to eliminate noise 
# reverses a dilation
eroded = cv2.erode(img, (3,3), iterations=3)

# crop image
cropped = img[100:600, 600:900]

cv2.imshow("Cities", cropped)
cv2.waitKey(0)

100

### 6 - Image transformations

Rotations, resizing, flipping and cropping an image

In [ ]:
import numpy as np
import cv2


def rescaleFrame(frame, scale=0.5):
    width = int(frame.shape[1] * scale)
    height = int(frame.shape[0] * scale)

    newRes = (width, height)
    newImg = cv2.resize(frame, newRes) # resize an image with the new specified resolution
    return newImg


# translate image
def translateFrame(frame, x, y):
    transMat = np.float32(
        [[1,0,x],
        [0,1,y]]
    )

    # -ve x --> left
    # -ve y --> up
    # +ve x --> right
    # +ve y --> down
    dimensions = (int(frame.shape[1] * 1.5), int(frame.shape[0] * 1.5)) # specify the new canvas/frame size

    newFrame = cv2.warpAffine(frame, transMat, dimensions)
    return newFrame


# rotate image
def rotateFrame(frame, angle, rotPoint=None): # angle - in degrees
    (height,width) = frame.shape[:2]

    if rotPoint is None:
        rotPoint = (width // 2, height // 2) # rotate at the center of the image

    rotMatrix = cv2.getRotationMatrix2D(rotPoint, angle, 1.0)
    dimensions = (width, height)
    return cv2.warpAffine(img, rotMatrix, dimensions)


# resize image
def resizeFrame(frame, width, height):
    resized = cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)
    return resized


# flip image
def flipFrame(frame):
    '''
    flip codes
    0 - vertical (reflect in the x-axis)
    1 - horizontal (reflect in the y-axis) 
    -1 - reflect both axes
    '''
    flipped = cv2.flip(frame, 1)
    return flipped

img = cv2.imread("test_img_city1.jpg")
img = rescaleFrame(img)

# img = translateFrame(img, 100, 100)
# img = rotateFrame(img, 90)
# img = resizeFrame(img, 500, 200)

cv2.imshow("Image", img)
cv2.waitKey(0)

100

: 

### 6 - Contour detection

Contours are continuous lines tracing the outer boundaries of an object.<br>
They are used to connect together edges (individual pixels showing a sudden change in colour).

In [ ]:
import cv2

def rescaleFrame(frame, scale=0.5):
    width = int(frame.shape[1] * scale)
    height = int(frame.shape[0] * scale)

    newRes = (width, height)
    newImg = cv2.resize(frame, newRes) # resize an image with the new specified resolution
    return newImg

img = cv2.imread("test_img_city1.jpg")
img = rescaleFrame(img)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

blur = cv2.GaussianBlur(gray, (5,5), sigmaX=0, borderType=cv2.BORDER_DEFAULT)
canny = cv2.Canny(blur, 125, 250)

# hierarchies - tells us the relationship between contours (e.g. which contours are inside other contours)
# contours are stored within a list, where each contour is a numpy array of (x,y) coordinates of the boundary points of the object
# they also have sibling (prev, next) and parent-child relationships (e.g. a contour can be inside another contour)
# very useful for storing the spatial context of the contours in the image

# RETR_EXTERNAL - onyl shows the outermost contours. children and inner contours ignored
# RETR_LIST - gets all contours but flattens the hierarchy (so no parent-child relationships, they are all the same level)
# RETR_CCOMP - gets all contours and organizes them into a two-level hierarchy (parent-child relationships)
# RETR_TREE - shows a full tree of related contours describing each level the hierarchy

# cv2.CHAIN_APPROX_SIMPLE + cv2.CHAIN_APPROX_NONE
contours, hierarchies = cv2.findContours(canny, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE) # only supports 32-bit signed int images. no colour
print(f"Found {len(contours)} contours in the image")

cv2.imshow("Image", canny)
cv2.waitKey(0)

Found 795 contours in the image
